# Data Exploration & Analysis

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
q3_2024 = pd.read_csv('../../data/2024_q3.csv')
q4_2024 = pd.read_csv('../../data/2024_q4.csv')
q1_2025 = pd.read_csv('../../data/2025_q1.csv')
q2_2025 = pd.read_csv('../../data/2025_q2.csv')

In [3]:
print(q3_2024.shape)
print(q4_2024.shape)
print(q1_2025.shape)
print(q2_2025.shape)

(8297869, 15)
(8525077, 15)
(7297028, 15)
(8450420, 15)


In [4]:
print(q3_2024.head())

   YEAR  QUARTER  ORIGIN_AIRPORT_ID ORIGIN  DEST_AIRPORT_ID DEST  \
0  2024        3              10135    ABE            10136  ABI   
1  2024        3              10135    ABE            10140  ABQ   
2  2024        3              10135    ABE            10140  ABQ   
3  2024        3              10135    ABE            10140  ABQ   
4  2024        3              10135    ABE            10140  ABQ   

  REPORTING_CARRIER TICKET_CARRIER OPERATING_CARRIER  BULK_FARE  PASSENGERS  \
0                MQ             AA                99        0.0         1.0   
1                9E             DL                99        0.0         1.0   
2                9E             DL                99        0.0         1.0   
3                9E             DL                99        0.0         1.0   
4                9E             DL                99        0.0         1.0   

   MARKET_FARE  MARKET_DISTANCE  NONSTOP_MILES  MKT_GEO_TYPE  
0        430.5           1575.0         1458.0       

In [5]:
q3_2024.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8297869 entries, 0 to 8297868
Data columns (total 15 columns):
 #   Column             Dtype  
---  ------             -----  
 0   YEAR               int64  
 1   QUARTER            int64  
 2   ORIGIN_AIRPORT_ID  int64  
 3   ORIGIN             object 
 4   DEST_AIRPORT_ID    int64  
 5   DEST               object 
 6   REPORTING_CARRIER  object 
 7   TICKET_CARRIER     object 
 8   OPERATING_CARRIER  object 
 9   BULK_FARE          float64
 10  PASSENGERS         float64
 11  MARKET_FARE        float64
 12  MARKET_DISTANCE    float64
 13  NONSTOP_MILES      float64
 14  MKT_GEO_TYPE       int64  
dtypes: float64(5), int64(5), object(5)
memory usage: 949.6+ MB


In [6]:
df = q3_2024.copy()

# Boolean column: True if same, False otherwise
same_carrier = df['TICKET_CARRIER'] == df['OPERATING_CARRIER']

# Count how many are the same
num_same = same_carrier.sum()

# Total rows
total = len(df)

# Percentage of matches
pct_same = num_same / total * 100

print(f"Same carrier count: {num_same:,}")
print(f"Total rows: {total:,}")
print(f"Percentage same: {pct_same:.2f}%")

Same carrier count: 6,245,425
Total rows: 8,297,869
Percentage same: 75.27%


In [7]:
q3_2024['YEAR'].max()

np.int64(2024)

In [8]:
q3_2024['MKT_GEO_TYPE'].value_counts()

MKT_GEO_TYPE
2    7755625
1     542244
Name: count, dtype: int64

In [9]:
len(q3_2024['REPORTING_CARRIER'].value_counts())

25

In [10]:
q3_2024.columns

Index(['YEAR', 'QUARTER', 'ORIGIN_AIRPORT_ID', 'ORIGIN', 'DEST_AIRPORT_ID',
       'DEST', 'REPORTING_CARRIER', 'TICKET_CARRIER', 'OPERATING_CARRIER',
       'BULK_FARE', 'PASSENGERS', 'MARKET_FARE', 'MARKET_DISTANCE',
       'NONSTOP_MILES', 'MKT_GEO_TYPE'],
      dtype='object')

In [11]:
q4_2024.columns

Index(['YEAR', 'QUARTER', 'ORIGIN_AIRPORT_ID', 'ORIGIN', 'DEST_AIRPORT_ID',
       'DEST', 'REPORTING_CARRIER', 'TICKET_CARRIER', 'OPERATING_CARRIER',
       'BULK_FARE', 'PASSENGERS', 'MARKET_FARE', 'MARKET_DISTANCE',
       'NONSTOP_MILES', 'MKT_GEO_TYPE'],
      dtype='object')

In [12]:
q1_2025.columns

Index(['YEAR', 'QUARTER', 'ORIGIN_AIRPORT_ID', 'ORIGIN', 'DEST_AIRPORT_ID',
       'DEST', 'REPORTING_CARRIER', 'TICKET_CARRIER', 'OPERATING_CARRIER',
       'BULK_FARE', 'PASSENGERS', 'MARKET_FARE', 'MARKET_DISTANCE',
       'NONSTOP_MILES', 'MKT_GEO_TYPE'],
      dtype='object')

In [13]:
df = pd.concat([q3_2024, q4_2024, q1_2025, q2_2025], ignore_index=True)

df.head()

,YEAR,QUARTER,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,REPORTING_CARRIER,TICKET_CARRIER,OPERATING_CARRIER,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE
0,2024,3,10135,ABE,10136,ABI,MQ,AA,99,0.0,1.0,430.5,1575.0,1458.0,2
1,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,5.5,1961.0,1738.0,2
2,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,148.5,1961.0,1738.0,2
3,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,182.0,1961.0,1738.0,2
4,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,309.5,1961.0,1738.0,2


In [14]:
df.shape

(32570394, 15)

In [15]:
airport_carrier = (
    df.groupby(['ORIGIN', 'OPERATING_CARRIER'])['PASSENGERS']
      .sum()
      .reset_index()
)

# Step 2: compute market share at each airport
airport_carrier['share'] = (
    airport_carrier['PASSENGERS'] /
    airport_carrier.groupby('ORIGIN')['PASSENGERS'].transform('sum')
)

# Step 3: compute HHI
airport_hhi = (
    airport_carrier.groupby('ORIGIN')['share']
      .apply(lambda x: (x**2).sum() * 10000)
      .reset_index(name='HHI')
)

print(airport_hhi.head())

  ORIGIN          HHI
0    ABE  4624.492464
1    ABI  7720.551144
2    ABQ  3012.014710
3    ABR  6811.831792
4    ABY  8931.553103


In [16]:
airport_hhi.to_csv('../../data/airport_hhi.csv', index=False)

In [17]:
df['OPERATING_CARRIER'].value_counts()

OPERATING_CARRIER
WN    6893395
99    5191244
DL    4332283
AA    3929760
UA    3498767
OO    1098521
NK    1062158
AS    1023397
F9     980306
YX     746790
B6     713049
G4     628275
MQ     436597
OH     391552
9E     363793
HA     257923
MX     225667
QX     192706
YV     168528
PT      89681
XP      86378
G7      79331
C5      73619
SY      60547
ZW      24756
3M      16317
--       3899
LF        505
9K        310
KG        141
9X        118
VC         58
AC          5
4B          4
7H          3
X4          3
LX          2
NZ          1
TK          1
LH          1
ET          1
TP          1
AN          1
Name: count, dtype: int64

In [18]:
df['TICKET_CARRIER'].value_counts()

TICKET_CARRIER
AA    7521707
WN    6893395
DL    6400641
UA    5596984
AS    1596444
NK    1062158
F9     980306
B6     720040
G4     628275
99     518221
HA     263107
MX     225667
XP      86378
SY      60547
3M      11381
--       3899
LF        505
KG        141
9X        118
QR        103
PR         96
9K         80
VC         58
RJ         30
QF         19
BA         12
IB         10
FJ          8
SQ          8
AC          7
CX          6
4B          4
EI          4
VS          3
AY          3
7H          3
X4          3
LH          3
NZ          3
AD          2
LA          2
LX          2
AT          1
LY          1
KE          1
TK          1
VA          1
ET          1
KL          1
AM          1
TP          1
AN          1
JL          1
Name: count, dtype: int64

In [19]:
df["REPORTING_CARRIER"].value_counts()

REPORTING_CARRIER
WN    6918441
DL    4929873
AA    4860444
UA    4049978
OO    2126803
AS    1150591
NK    1062158
F9     980307
YX     963134
MQ     885731
OH     757320
B6     735473
9E     656363
G4     628275
PT     286230
QX     284370
HA     272816
YV     257519
MX     225667
C5     157852
G7     138459
XP      86378
ZW      76705
SY      61200
3M      18307
Name: count, dtype: int64

In [20]:
df = df[df['REPORTING_CARRIER'] != '99']
df = df[df['TICKET_CARRIER'] != '99']
df = df[df['OPERATING_CARRIER'] != '99']

In [21]:
df.shape

(27378923, 15)

In [22]:
same_carrier = df['TICKET_CARRIER'] == df['OPERATING_CARRIER']

# Count how many are the same
num_same = same_carrier.sum()

# Total rows
total = len(df)

# Percentage of matches
pct_same = num_same / total * 100

print(f"Same carrier count: {num_same:,}")
print(f"Total rows: {total:,}")
print(f"Percentage same: {pct_same:.2f}%")

Same carrier count: 23,609,712
Total rows: 27,378,923
Percentage same: 86.23%


In [23]:
df[same_carrier == False][['TICKET_CARRIER', 'OPERATING_CARRIER']]

,TICKET_CARRIER,OPERATING_CARRIER
41,DL,9E
42,DL,9E
43,DL,9E
44,DL,9E
45,DL,9E
...,...,...
32570388,UA,OO
32570389,UA,OO
32570390,UA,OO
32570391,UA,OO


In [24]:
df[df['OPERATING_CARRIER']=='G7']['TICKET_CARRIER'].value_counts()

TICKET_CARRIER
UA    79331
Name: count, dtype: int64

In [25]:
df[df['MKT_GEO_TYPE'] == 1].head()

,YEAR,QUARTER,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,REPORTING_CARRIER,TICKET_CARRIER,OPERATING_CARRIER,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE
8181,2024,3,10140,ABQ,10170,ADQ,AS,AS,AS,0.0,1.0,5.50,2881.0,2619.0,1
8182,2024,3,10140,ABQ,10170,ADQ,AS,AS,AS,0.0,1.0,144.00,2881.0,2619.0,1
8399,2024,3,10140,ABQ,10299,ANC,AA,AA,AA,0.0,1.0,5.06,3612.0,2618.0,1
8400,2024,3,10140,ABQ,10299,ANC,AA,AA,AA,0.0,1.0,274.00,3612.0,2618.0,1
8401,2024,3,10140,ABQ,10299,ANC,AA,AA,AA,0.0,1.0,383.00,3612.0,2618.0,1
